# 03 Feature Engineering and Supervised Modeling

**Project:** STAT GR5243 Project 4: End-to-End Machine Learning  
**Topic:** Predicting High-Occupancy Airbnb Listings in New York City

This notebook builds supervised classification models to predict whether an Airbnb listing is high-occupancy.

## 1. Import Libraries and Load Data
Run this notebook from the `notebooks/` folder.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, classification_report
)

processed_path = Path("../data/processed")
fig_path = Path("../figures")
fig_path.mkdir(parents=True, exist_ok=True)

clustered_file = processed_path / "nyc_airbnb_cleaned_with_clusters.csv"
cleaned_file = processed_path / "nyc_airbnb_cleaned.csv"

if clustered_file.exists():
    df = pd.read_csv(clustered_file, low_memory=False)
    print("Loaded:", clustered_file)
else:
    df = pd.read_csv(cleaned_file, low_memory=False)
    print("Loaded:", cleaned_file)

print("Data shape:", df.shape)
df.head()

## 2. Define Target and Remove Leakage-Prone Variables

The target is `high_occupancy`. Since it was created from `estimated_occupancy_l365d`, we remove that variable from predictors.

We also remove review-count and availability variables from the main model because they may act as post-outcome proxies for booking activity. This makes the model more conservative and defensible.

In [ ]:
target = "high_occupancy"

identifier_cols = ["id", "listing_url", "host_since"]

direct_target_cols = [
    "high_occupancy",
    "estimated_occupancy_l365d"
]

proxy_cols = [
    "availability_30", "availability_60", "availability_90", "availability_365", "availability_eoy",
    "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d", "number_of_reviews_ly",
    "reviews_per_month",
    "pca1", "pca2"
]

drop_cols = [c for c in identifier_cols + direct_target_cols + proxy_cols if c in df.columns]

X = df.drop(columns=drop_cols)
y = df[target].astype(int)

print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts(normalize=True))
print("Dropped columns:", drop_cols)

## 3. Feature Engineering: Numeric and Categorical Preprocessing

Categorical variables are one-hot encoded. Numeric variables are standardized.

In [ ]:
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32", "bool"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Number of numeric features:", len(numeric_features))

try:
    sparse_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    dense_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    sparse_encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)
    dense_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

sparse_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", sparse_encoder, categorical_features)
    ]
)

dense_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", dense_encoder, categorical_features)
    ]
)

## 4. Train/Test Split
We use a stratified split so that the high-occupancy share is similar in train and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

## 5. Define Three Supervised Models

We compare:

1. Logistic Regression: interpretable baseline  
2. Random Forest: nonlinear tree-based ensemble  
3. Histogram Gradient Boosting: stronger boosting model for nonlinear patterns

In [ ]:
models = {
    "Logistic Regression": Pipeline(steps=[
        ("preprocess", sparse_preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocess", sparse_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=150,
            max_depth=14,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "Hist Gradient Boosting": Pipeline(steps=[
        ("preprocess", dense_preprocessor),
        ("model", HistGradientBoostingClassifier(
            max_iter=150,
            learning_rate=0.06,
            max_leaf_nodes=31,
            random_state=42
        ))
    ])
}

## 6. Train and Evaluate Models on Test Set

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    return {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score)
    }, y_pred, y_score

results = []
predictions = {}
scores = {}

for name, model in models.items():
    print("Training:", name)
    metrics, y_pred, y_score = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    results.append(metrics)
    predictions[name] = y_pred
    scores[name] = y_score

model_comparison = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
model_comparison

## 7. Cross-Validation

We use 3-fold stratified cross-validation to check performance stability.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

cv_results = []

for name, model in models.items():
    print("Cross-validating:", name)
    scores_cv = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
        n_jobs=-1
    )
    cv_results.append({
        "model": name,
        "cv_accuracy_mean": scores_cv["test_accuracy"].mean(),
        "cv_precision_mean": scores_cv["test_precision"].mean(),
        "cv_recall_mean": scores_cv["test_recall"].mean(),
        "cv_f1_mean": scores_cv["test_f1"].mean(),
        "cv_roc_auc_mean": scores_cv["test_roc_auc"].mean(),
        "cv_roc_auc_std": scores_cv["test_roc_auc"].std()
    })

cv_comparison = pd.DataFrame(cv_results).sort_values("cv_roc_auc_mean", ascending=False)
cv_comparison

## 8. Save Model Comparison Tables

In [ ]:
model_comparison.to_csv(processed_path / "model_comparison_test_metrics.csv", index=False)
cv_comparison.to_csv(processed_path / "model_comparison_cv_metrics.csv", index=False)

print("Saved model comparison tables to data/processed.")

## 9. Final Model Selection, Confusion Matrix, and ROC Curve

The final model is selected by ROC-AUC first, then F1-score and interpretability.

In [ ]:
final_model_name = model_comparison.iloc[0]["model"]
final_pred = predictions[final_model_name]
final_score = scores[final_model_name]

print("Selected final model:", final_model_name)
print(classification_report(y_test, final_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, final_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title(f"Confusion Matrix: {final_model_name}")
plt.tight_layout()
plt.savefig(fig_path / "12_final_model_confusion_matrix.png", dpi=200)
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(y_test, final_score)
plt.title(f"ROC Curve: {final_model_name}")
plt.tight_layout()
plt.savefig(fig_path / "13_final_model_roc_curve.png", dpi=200)
plt.show()

## 10. Feature Importance from Random Forest

Even if Random Forest is not the final model, it gives useful feature-importance interpretation.

In [ ]:
rf_pipeline = models["Random Forest"]
rf_pipeline.fit(X_train, y_train)

preprocessor = rf_pipeline.named_steps["preprocess"]
rf_model = rf_pipeline.named_steps["model"]

try:
    feature_names = preprocessor.get_feature_names_out()
except Exception:
    feature_names = [f"feature_{i}" for i in range(len(rf_model.feature_importances_))]

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

importance_df.to_csv(processed_path / "random_forest_feature_importance.csv", index=False)
importance_df.head(20)

In [ ]:
top_features = importance_df.head(15).sort_values("importance", ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.title("Top Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(fig_path / "14_random_forest_feature_importance.png", dpi=200)
plt.show()

## 11. Modeling Summary

Main takeaways:

1. We framed the supervised task as binary classification.
2. We removed direct target variables and several post-outcome proxy variables to reduce leakage.
3. We compared Logistic Regression, Random Forest, and Histogram Gradient Boosting.
4. We evaluated models using Accuracy, Precision, Recall, F1-score, and ROC-AUC.
5. The final model was selected based on predictive performance and interpretability.